# MIDA507 -- Notebook Session 03
## Open Data, Licences et Portails Africains

| | |
|---|---|
| **Session** | S03 -- Open Data, licences et portails africains |
| **Prerequis** | Notebooks S01 et S02 |
| **Duree** | 60 a 80 minutes |

### Objectifs
1. Comprendre les 10 principes Sunlight de l'open data
2. Distinguer les licences ouvertes (CC0, CC-BY, CC-BY-SA, Etalab)
3. Auditer des portails open data africains (data.gouv.bj, opendata.sn, INS-CM)
4. Identifier les 8 barrieres africaines a l'open data et leurs solutions IDA
5. Produire une fiche d'audit de portail

### Livrable : `MIDA507_S03_audit_portails_africains.json`


---
## PARTIE 1 -- Configuration


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as mpatches, seaborn as sns
import json, os, random, hashlib
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
PAL = {"ter1":"#3E1500","ter2":"#BF360C","ter3":"#E64A19",
       "olive":"#558B2F","bleu":"#1565C0","gris":"#546E7A","clair":"#FBE9E7"}

def regen_s02(n=5000):
    random.seed(42); np.random.seed(42)
    genres  = ["These et memoire","Manuel universitaire","Revue scientifique",
               "Rapport de recherche","Ouvrage de reference","Document administratif"]
    fil     = ["Droit","Sciences economiques","Lettres modernes","Histoire",
               "Geographie","Medecine","Agronomie","Informatique"]
    niv     = ["Licence 1","Licence 2","Licence 3","Master 1","Master 2","Doctorat"]
    reg     = ["Adamaoua","Centre","Est","Extreme-Nord","Littoral","Nord","Ouest","Sud"]
    lang    = ["Francais","Anglais","Bilingue FR/EN","Arabe","Autres langues africaines"]
    langiso = ["fra","eng","fra+eng","ara","mul"]
    d0 = datetime(2018,1,1)
    dates = [(d0+timedelta(days=random.randint(0,365*5))).strftime("%Y-%m-%d") for _ in range(n)]
    df = pd.DataFrame({
        "ark_id":[f"ark:/67375/UNG-{str(i+1).zfill(6)}" for i in range(n)],
        "cote":[f"UNG-{str(i+1).zfill(5)}" for i in range(n)],
        "genre_document":np.random.choice(genres,n,p=[0.28,0.30,0.15,0.10,0.10,0.07]),
        "date_emprunt":dates,
        "duree_jours":np.random.choice([3,7,14,21,30],n,p=[0.1,0.3,0.3,0.2,0.1]),
        "code_usager_anon":[f"USR-{hex(random.randint(0,0xFFFFFF))[2:].upper().zfill(6)}" for _ in range(n)],
        "filiere":np.random.choice(fil,n),
        "niveau":np.random.choice(niv,n),
        "region_origine":np.random.choice(reg,n),
        "langue_document":np.random.choice(lang,n,p=[0.62,0.22,0.10,0.04,0.02]),
        "langue_iso639":np.random.choice(langiso,n,p=[0.62,0.22,0.10,0.04,0.02]),
    })
    df["annee"] = pd.to_datetime(df["date_emprunt"]).dt.year
    return df

CSV = "MIDA507_S02_bibliotheque_ung_enrichie.csv"
if os.path.exists(CSV):
    df = pd.read_csv(CSV)
    print(f"Jeu S02 charge : {len(df):,} lignes")
else:
    df = regen_s02()
    print(f"Jeu S02 regenere : {len(df):,} lignes")
print(f"Colonnes : {list(df.columns)}")


---
## PARTIE 2 -- Les 10 Principes Sunlight de l'Open Data

> Les principes Sunlight (2007) sont le referentiel mondial de l'open data gouvernemental.
> L'IDA les utilise pour auditer et ameliorer les portails documentaires africains.


In [ ]:
PRINCIPES = [
    ("1. Completude",       "Jeux de donnees complets -- aucune donnee collectee n'est omise arbitrairement.",
     "INS Cameroun publie ses agregats mais pas les microdonnes brutes par menage."),
    ("2. Primaute",         "Donnees publiees aussi proches que possible de leur source originale.",
     "BU-UNG publiera ses donnees depuis le SIGB PMB sans agregation prealable."),
    ("3. Actualite",        "Donnees publiees des que possible avec frequence de MAJ declaree.",
     "ANSD Senegal : indicateurs annuels publies 3 mois apres la fin de l'exercice."),
    ("4. Accessibilite",    "Donnees accessibles au plus grand nombre sans inscription ni paiement.",
     "data.gouv.bj : telechargement direct CSV sans compte, en HTTPS."),
    ("5. Lisibilite machine","Donnees structurees pour le traitement auto (CSV, JSON) -- pas PDF.",
     "BUCREP Cameroun : publication en Excel uniquement -- non conforme."),
    ("6. Non-discrimination","Donnees accessibles a tous sans discrimination.",
     "Pas de difference d'acces entre chercheurs nationaux et diaspora africaine."),
    ("7. Non-propriete",    "Formats ouverts non controles par une entite unique (CSV, ODS, JSON).",
     "Refus des formats XLS, DOC, PDF proprietaires comme format principal."),
    ("8. Licence libre",    "Licence autorisant la reutilisation, modification et redistribution.",
     "CC-BY 4.0 -- standard recommande pour les donnees africaines."),
    ("9. Permanence",       "Donnees disponibles en ligne avec gestion des versions et archivage.",
     "Identifiant ARK garanti permanent meme si l'URL du portail change."),
    ("10. Usage payant nul","Donnees gratuites -- pas de tarif meme pour usage commercial.",
     "CEDEAO : certains rapports economiques payants = violation du principe 10."),
]

# Tableau HTML
from IPython.display import display, HTML
rows = ""
for i,(nom,defn,ex) in enumerate(PRINCIPES):
    bg = "#FBE9E7" if i%2==0 else "white"
    rows += f"""<tr style="background:{bg}">
      <td style="padding:6px 10px;font-weight:bold;color:#E64A19">{nom}</td>
      <td style="padding:6px 10px">{defn}</td>
      <td style="padding:6px 10px;font-style:italic;color:#666">{ex}</td>
    </tr>"""
html = f"""<table style="width:100%;border-collapse:collapse;font-size:12px;font-family:Arial">
<tr style="background:#3E1500;color:white">
  <th style="padding:7px 10px">Principe</th>
  <th style="padding:7px 10px">Definition</th>
  <th style="padding:7px 10px">Exemple africain</th>
</tr>{rows}</table>"""
display(HTML(html))
print("\n Notre jeu BU-UNG respectera 9 principes sur 10 apres les sessions MIDA507.")


---
## PARTIE 3 -- Licences Ouvertes : choisir la bonne licence

> Sans licence, toute reutilisation est juridiquement impossible -- meme si le fichier est en ligne.


In [ ]:
LICENCES = [
    ("CC0",     "Domaine public -- aucune restriction",        "#546E7A",
     "Donnees meteo INM -- aucune revendication d'auteur"),
    ("CC-BY",   "Attribution seule requise",                  "#3E1500",
     "data.gouv.bj -- standard recommande Afrique francophone"),
    ("CC-BY-SA","Attribution + partage a l'identique",        "#BF360C",
     "OpenStreetMap Africa -- repartage obligatoire sous CC-BY-SA"),
    ("CC-BY-NC","Attribution + usage non commercial",         "#FF8A65",
     "Deconseille pour l'open data gouvernemental"),
    ("Etalab",  "Licence gouvernementale francaise",           "#558B2F",
     "ANSD Senegal, ONECI Cote d'Ivoire"),
]

fig, axes = plt.subplots(1, 5, figsize=(17, 7))
for ax, (nom, desc, couleur, ex) in zip(axes, LICENCES):
    ax.set_xlim(0,10); ax.set_ylim(0,10); ax.axis("off")
    ax.add_patch(plt.FancyBboxPatch((0.3,7.5),9.4,2.2,boxstyle="round,pad=0.15",
                 facecolor=couleur,edgecolor="white",linewidth=2))
    ax.text(5,8.7,nom,ha="center",va="center",fontsize=14,fontweight="bold",color="white")
    ax.text(5,6.8,desc,ha="center",va="center",fontsize=8,color="#333",style="italic")
    ax.text(5,5.5,"Exemple :",ha="center",fontsize=8,fontweight="bold",color=couleur)
    ax.text(5,4.3,ex,ha="center",va="center",fontsize=8,color="#555",style="italic")
    # Barres de droits
    droits_labels = ["Usage libre","Usage commercial","Modification","Attribution"]
    droits_valeurs = {"CC0":[1,1,1,0],"CC-BY":[1,1,1,1],
                      "CC-BY-SA":[1,1,1,1],"CC-BY-NC":[1,0,1,1],"Etalab":[1,1,1,1]}[nom]
    for j,(lab,val) in enumerate(zip(droits_labels,droits_valeurs)):
        col = "#2E7D32" if val else "#CC0000"
        sym = "✅" if val else "❌"
        ax.text(0.5, 2.8-j*0.8, f"{sym} {lab}", fontsize=7.5, color=col)

plt.suptitle("Licences Ouvertes -- Guide de choix pour l'IDA africain",
             fontsize=13,fontweight="bold",color="#3E1500",y=1.02)
plt.tight_layout()
plt.show()
print("RECOMMANDATION IDA pour la BU-UNG : Licence CC-BY 4.0")
print("La plus permissive avec attribution -- maximise la reutilisation")
print("par les chercheurs, ONG et journalistes identifies en S02.")


---
## PARTIE 4 -- Audit Comparatif des Portails Africains


In [ ]:
PORTAILS = {
    "data.gouv.bj": {"pays":"Benin","plateforme":"CKAN","api":True,"https":True,"inscription":False,
        "sunlight":[8,8,7,10,9,10,9,9,7,10],"licence":"Etalab v2.0","formats":["CSV","JSON","XLS"],
        "commentaire":"Reference francophone africaine. API CKAN documentee."},
    "opendata.sn":  {"pays":"Senegal","plateforme":"CKAN","api":True,"https":True,"inscription":False,
        "sunlight":[6,7,6,8,8,9,8,7,6,10],"licence":"Conditions ANSD","formats":["CSV","XLS","PDF"],
        "commentaire":"Bonne couverture statistique. Licence non standard."},
    "ins.cm_hypoth":{"pays":"Cameroun (hypothetique)","plateforme":"Portail custom","api":False,"https":False,"inscription":True,
        "sunlight":[4,4,3,5,4,7,4,2,3,8],"licence":"Aucune","formats":["XLS","PDF"],
        "commentaire":"Situation typique d'un portail statistique africain non optimise."},
}

categories = ["Completude","Primaute","Actualite","Accessibilite","Lisibilite",
              "Non-discrim.","Non-propriete","Licence","Permanence","Gratuit"]
N = len(categories)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]
couleurs_p = ["#3E1500","#1565C0","#CC0000"]

fig, ax = plt.subplots(figsize=(9,9), subplot_kw=dict(projection="polar"))
for (nom, data), col in zip(PORTAILS.items(), couleurs_p):
    vals = data["sunlight"] + data["sunlight"][:1]
    ax.plot(angles, vals, "o-", lw=2, label=f"{nom}", color=col)
    ax.fill(angles, vals, alpha=0.08, color=col)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=9)
ax.set_ylim(0,10); ax.set_yticks([2,4,6,8,10])
ax.set_title("Audit Sunlight -- Portails Open Data Africains (Score /10 par principe)",
             fontsize=12, fontweight="bold", color="#3E1500", pad=18)
ax.legend(loc="upper right", bbox_to_anchor=(1.4,1.1), fontsize=9)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout(); plt.show()

print("\nSCORES GLOBAUX SUNLIGHT (/100)")
print("="*50)
for nom, data in PORTAILS.items():
    s = sum(data["sunlight"])
    niv = "OK" if s>=75 else "Partiel" if s>=50 else "Insuffisant"
    print(f"  {nom:<35} {s}/100  ({niv})")
print("\nPays reference : data.gouv.bj avec 87/100 -- modele pour l'Afrique francophone")


---
## PARTIE 5 -- Les 8 Barrieres a l'Open Data en Afrique


In [ ]:
BARRIERES = [
    ("1. Culturelle",       "Mentalite 'les donnees sont un actif a conserver'",
     "Formation des cadres, montrer la valeur de la publication avec des cas africains"),
    ("2. Technique",        "Absence d'infrastructure, connexions lentes, manque de competences",
     "CKAN heberge, Google Sheets + API, formation IDA data steward"),
    ("3. Juridique",        "Vide legislatif sur l'open data, lois obsoletes",
     "Adopter CC-BY / Etalab meme sans cadre legal national specifique"),
    ("4. Financiere",       "Absence de budget pour la mise en ligne et la maintenance",
     "CKAN gratuit, Zenodo gratuit (5 Go), GitHub pour petits jeux de donnees"),
    ("5. Qualite",          "Donnees incompletes, formats heterogenes, valeurs manquantes",
     "OpenRefine + Python (sessions S01-S05), politique de qualite avant publication"),
    ("6. Interoperabilite", "Jeux non connectables entre eux, pas d'IDs communs",
     "ARK persistants, DCAT, AGROVOC/RAMEAU, ISO 639 pour les langues"),
    ("7. Politique",        "Opposition de certains ministeres aux donnees sensibles",
     "Publier les donnees agreges et anonymisees -- confidentialite et open data compatibles"),
    ("8. Durabilite",       "Portails abandonnes faute de maintenance, URLs mortes",
     "ARK + Zenodo pour preservation longue duree, politique de maintenance dans le DMP"),
]

fig, axes = plt.subplots(2, 4, figsize=(18,10))
axes = axes.flatten()
cols = ["#3E1500","#BF360C","#E64A19","#FF7043","#558B2F","#1565C0","#4A148C","#546E7A"]
for ax,(nom,prob,sol),col in zip(axes, BARRIERES, cols):
    ax.set_xlim(0,10); ax.set_ylim(0,8); ax.axis("off")
    ax.add_patch(plt.FancyBboxPatch((0.1,6.2),9.8,1.5,boxstyle="round,pad=0.1",
                 facecolor=col,edgecolor="white",linewidth=2))
    ax.text(5,7.0,nom,ha="center",va="center",fontsize=10,fontweight="bold",color="white")
    ax.text(0.4,5.5,"Probleme :",fontsize=8.5,fontweight="bold",color="#CC0000")
    ax.text(0.4,4.7,prob,fontsize=8.5,color="#333",wrap=True)
    ax.text(0.4,3.4,"Solution IDA :",fontsize=8.5,fontweight="bold",color="#2E7D32")
    ax.text(0.4,2.0,sol,fontsize=8.5,color="#2E7D32",wrap=True)

plt.suptitle("8 Barrieres a l'Open Data en Afrique et Solutions IDA",
             fontsize=14,fontweight="bold",color="#3E1500",y=1.01)
plt.tight_layout(); plt.show()


---
### EXERCICE -- Auditez un portail africain avec les 10 principes Sunlight


In [ ]:
# ============================================================
# EXERCICE -- Complétez l'audit d'un portail africain réel
# ============================================================
# Choisissez un portail open data africain et notez chaque principe de 0 à 10.

MON_PORTAIL   = "[Nom du portail]"   # <- changez ici
MON_PAYS      = "[Pays]"             # <- changez ici

NOTES = {
    "completude":         None,  # <- 0 a 10
    "primaute":           None,
    "actualite":          None,
    "accessibilite":      None,
    "lisibilite_machine": None,
    "non_discrimination": None,
    "non_propriete":      None,
    "licence_libre":      None,
    "permanence":         None,
    "usage_payant_nul":   None,
}
COMMENTAIRE = "[Votre observation principale]"

# Calcul automatique
valides = {k:v for k,v in NOTES.items() if isinstance(v,(int,float)) and 0<=v<=10}
score = sum(valides.values())
maxi  = len(valides)*10

print(f"AUDIT SUNLIGHT -- {MON_PORTAIL} ({MON_PAYS})")
print("="*55)
for p,n in NOTES.items():
    if n is None:
        print(f"  [?] {p:<28} : Non note")
    else:
        bar = chr(9608)*int(n) + chr(9617)*(10-int(n))
        em  = "OK" if n>=7 else ("~" if n>=4 else "X")
        print(f"  [{em}] {p:<28} : {n}/10  {bar}")

if valides:
    pct = int(score/maxi*100) if maxi else 0
    print(f"\n  Score global : {score}/{maxi} ({pct}%)")
    print(f"  Niveau : {'Excellent' if pct>=80 else 'Conforme' if pct>=65 else 'Partiel' if pct>=50 else 'Insuffisant'}")
    audit = {"portail":MON_PORTAIL,"pays":MON_PAYS,"date":datetime.now().strftime("%Y-%m-%d"),
             "notes":NOTES,"score":score,"commentaire":COMMENTAIRE}
    with open("MIDA507_S03_audit_portails_africains.json","w",encoding="utf-8") as f:
        json.dump(audit,f,ensure_ascii=False,indent=2)
    print("\nAudit sauvegarde : MIDA507_S03_audit_portails_africains.json")
else:
    print("\n[A COMPLETER] Entrez des notes 0-10 pour chaque principe pour calculer le score.")


---
## Bilan S03 -- Ce que vous avez accompli

| Competence | Statut |
|---|---|
| Connaitre les 10 principes Sunlight | ✅ |
| Choisir une licence open data adaptee | ✅ |
| Auditer et comparer des portails africains | ✅ |
| Identifier les 8 barrieres africaines | ✅ |
| Conduire un audit Sunlight sur un portail reel | 🟡 A completer |

**Lien S04 :** Le jeu BU-UNG sera equipe d'une fiche DCAT complete et publiee sur CKAN.

*Notebook MIDA507 S03 -- Master MIDA -- Universite de Douala*
